# Setup Detectors v2 — BASE / VCP / PIVOT

Clean reimplementation of the agreed detection system.

**Detectors:**
1. Swing tiers (major + minor)
2. Simplified BASE detector
3. VCP detector (2-window with 30% contraction rule)
4. PIVOT detector (3-day low-vol + tight range)
5. Interactive chart with all overlays

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dataclasses import dataclass, field
from typing import Literal

## 1 — Fetch Data

In [2]:
TICKER = "APLD"
PERIOD = "2y"
INTERVAL = "1d"

raw = yf.Ticker(TICKER).history(period=PERIOD, interval=INTERVAL)
if raw.empty:
    raise ValueError("No data returned from yfinance.")

if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

raw = raw[["Open", "High", "Low", "Close", "Volume"]].dropna().reset_index()
raw.rename(columns={"Date": "date", "Datetime": "date"}, inplace=True)

df = raw.rename(columns={
    "Open": "open", "High": "high", "Low": "low",
    "Close": "close", "Volume": "volume",
})

print(f"Loaded {TICKER}: {len(df)} bars")
df.tail(3)

Loaded APLD: 502 bars


,date,open,high,low,close,volume
499,2026-02-18 00:00:00-05:00,31.320000,33.450001,30.299999,31.910000,24045600
500,2026-02-19 00:00:00-05:00,31.280001,31.750000,30.563000,31.530001,15011800
501,2026-02-20 00:00:00-05:00,30.870001,31.290001,28.365000,29.040001,23796300


## 2 — Indicators

In [3]:
def true_range(df: pd.DataFrame) -> pd.Series:
    prev_close = df["close"].shift(1)
    tr1 = df["high"] - df["low"]
    tr2 = (df["high"] - prev_close).abs()
    tr3 = (df["low"] - prev_close).abs()
    return pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)


def atr_series(df: pd.DataFrame, period: int = 14) -> pd.Series:
    return true_range(df).rolling(window=period, min_periods=period).mean()


def average_bar_size(df: pd.DataFrame, period: int = 20) -> pd.Series:
    return (df["high"] - df["low"]).rolling(window=period, min_periods=1).mean()


df["atr14"] = atr_series(df, 14)
df["abs20"] = average_bar_size(df, 20)
df["ema20"] = df["close"].ewm(span=20, adjust=False).mean()
df["sma50"] = df["close"].rolling(50).mean()
df["sma200"] = df["close"].rolling(200).mean()
df["vol_avg20"] = df["volume"].rolling(20).mean()

print("Indicators computed.")
df[["close", "atr14", "abs20", "ema20", "sma50", "sma200", "vol_avg20"]].tail(3)

Indicators computed.


,close,atr14,abs20,ema20,sma50,sma200,vol_avg20
499,31.910000,4.057143,4.13560,34.697402,32.0569,21.30545,32006585.0
500,31.530001,3.922643,3.91095,34.395745,32.0631,21.43660,30649500.0
501,29.040001,3.824429,3.94420,33.885674,32.0017,21.55595,30385175.0


## 3 — Swing Point Detectors

In [4]:
@dataclass
class SwingPoint:
    index: int
    price: float
    type: Literal["HIGH", "LOW"]
    atr: float = 0.0
    prominence: float = 0.0


def detect_fractal_pivots(
    df: pd.DataFrame,
    lookahead: int = 10,
) -> list[SwingPoint]:
    """One-sided lookahead fractal pivots with ABS tolerance."""
    highs = df["high"].values
    lows = df["low"].values
    atr_vals = df["atr14"].values
    abs_vals = df["abs20"].values
    n = len(df)
    points: list[SwingPoint] = []

    for t in range(n - lookahead):
        abs_tol = abs_vals[t] if np.isfinite(abs_vals[t]) else 0
        a = atr_vals[t] if np.isfinite(atr_vals[t]) else abs_tol

        is_high = all(highs[i] < highs[t] - abs_tol for i in range(t + 1, t + lookahead + 1))
        if is_high:
            points.append(SwingPoint(index=t, price=highs[t], type="HIGH", atr=a))

        is_low = all(lows[i] > lows[t] + abs_tol for i in range(t + 1, t + lookahead + 1))
        if is_low:
            points.append(SwingPoint(index=t, price=lows[t], type="LOW", atr=a))

    return points


def detect_significant_swings(
    df: pd.DataFrame,
    left: int = 3,
    right: int = 3,
    atr_period: int = 14,
    prom_atr: float = 1.5,
    depart_atr: float = 2.5,
    depart_lookahead: int = 10,
    min_swing_sep: int = 7,
) -> list[SwingPoint]:
    """
    4-gate significant swing detection:
      (A) Fractal local extremum over [t-left .. t+right]
      (B) Prominence >= prom_atr * ATR
      (C) Departure after pivot >= depart_atr * ATR
      (D) Spacing dedup: keep most extreme within min_swing_sep bars
    """
    highs = df["high"].values
    lows = df["low"].values
    atr_vals = atr_series(df, atr_period).values
    n = len(df)
    candidates: list[SwingPoint] = []

    for t in range(left, n - right):
        a = atr_vals[t]
        if not np.isfinite(a) or a <= 0:
            continue

        window = slice(t - left, t + right + 1)

        # Pivot high
        if highs[t] == highs[window].max() and all(
            highs[i] < highs[t] for i in range(t - left, t + right + 1) if i != t
        ):
            local_min_low = lows[window].min()
            prominence = highs[t] - local_min_low
            if prominence >= prom_atr * a:
                dep_end = min(n, t + depart_lookahead + 1)
                min_low_after = lows[t + 1 : dep_end].min() if t + 1 < dep_end else np.inf
                if min_low_after <= highs[t] - depart_atr * a:
                    candidates.append(SwingPoint(t, highs[t], "HIGH", a, prominence))

        # Pivot low
        if lows[t] == lows[window].min() and all(
            lows[i] > lows[t] for i in range(t - left, t + right + 1) if i != t
        ):
            local_max_high = highs[window].max()
            prominence = local_max_high - lows[t]
            if prominence >= prom_atr * a:
                dep_end = min(n, t + depart_lookahead + 1)
                max_high_after = highs[t + 1 : dep_end].max() if t + 1 < dep_end else -np.inf
                if max_high_after >= lows[t] + depart_atr * a:
                    candidates.append(SwingPoint(t, lows[t], "LOW", a, prominence))

    # Spacing dedup
    candidates.sort(key=lambda p: p.index)
    result: list[SwingPoint] = []
    for p in candidates:
        if not result:
            result.append(p)
            continue
        last = result[-1]
        if p.type == last.type and p.index - last.index <= min_swing_sep:
            keep_new = p.price > last.price if p.type == "HIGH" else p.price < last.price
            if keep_new:
                result[-1] = p
        else:
            result.append(p)

    return result


print("Swing detectors defined.")

Swing detectors defined.


## 4 — Major / Minor Swing Tiers

In [5]:
major_swings = detect_significant_swings(
    df, left=3, right=3, prom_atr=1.5, depart_atr=2.5,
    depart_lookahead=10, min_swing_sep=7,
)
minor_swings = detect_significant_swings(
    df, left=2, right=2, prom_atr=0.8, depart_atr=1.2,
    depart_lookahead=6, min_swing_sep=3,
)

maj_h = sum(1 for s in major_swings if s.type == "HIGH")
maj_l = sum(1 for s in major_swings if s.type == "LOW")
min_h = sum(1 for s in minor_swings if s.type == "HIGH")
min_l = sum(1 for s in minor_swings if s.type == "LOW")
print(f"Major swings: {len(major_swings)} ({maj_h} highs, {maj_l} lows)")
print(f"Minor swings: {len(minor_swings)} ({min_h} highs, {min_l} lows)")

Major swings: 45 (22 highs, 23 lows)
Minor swings: 109 (58 highs, 51 lows)


## 5 — Simplified BASE Detector

In [6]:
def detect_base_simple(
    df: pd.DataFrame,
    swings: list[SwingPoint],
    min_base_duration: int = 20,
    max_retrace: float = 0.50,
) -> list[dict]:
    """
    Simplified BASE:
      - Start from significant swing HIGH
      - First window = peak to next lowest raw bar low
      - Retrace < 50%
      - No breakout above swing high during base
      - Drawdown <= 50%
      - Duration > 20 bars
    Returns one candidate per peak (latest valid end).
    """
    highs = df["high"].values
    lows = df["low"].values
    n = len(df)
    out: list[dict] = []

    peaks = sorted([s for s in swings if s.type == "HIGH"], key=lambda s: s.index)

    for pk in peaks:
        p = pk.index
        peak_price = float(pk.price)
        if p >= n - (min_base_duration + 1):
            continue

        best = None
        for t in range(p + min_base_duration + 1, n):
            span_high = float(np.max(highs[p : t + 1]))
            span_low = float(np.min(lows[p : t + 1]))

            low_idx = p + int(np.argmin(lows[p : t + 1]))
            low_price = float(lows[low_idx])

            retrace = (peak_price - low_price) / peak_price if peak_price > 0 else 1.0
            no_breakout = span_high <= peak_price
            no_overdrawdown = retrace < max_retrace
            duration = t - p + 1

            if no_breakout and no_overdrawdown and duration > min_base_duration:
                best = {
                    "kind": "BASE",
                    "peak_idx": int(p),
                    "peak_price": round(peak_price, 2),
                    "start_idx": int(p),
                    "low_idx": int(low_idx),
                    "end_idx": int(t),
                    "base_high": round(float(span_high), 2),
                    "base_low": round(float(span_low), 2),
                    "first_low_price": round(low_price, 2),
                    "retrace_pct": round(retrace, 4),
                    "duration": int(duration),
                }

        if best is not None:
            out.append(best)

    return out


base_raw = detect_base_simple(df, major_swings, min_base_duration=20, max_retrace=0.50)
print(f"BASE candidates: {len(base_raw)}")
for b in base_raw[:5]:
    print(f"  peak={b['peak_price']} low={b['first_low_price']} retrace={b['retrace_pct']:.3f} dur={b['duration']}")

BASE candidates: 6
  peak=7.23 low=3.93 retrace=0.456 dur=27
  peak=6.88 low=3.6 retrace=0.477 dur=33
  peak=11.25 low=6.12 retrace=0.456 dur=50
  peak=10.2 low=6.12 retrace=0.400 dur=29
  peak=15.42 low=9.02 retrace=0.415 dur=46


## 6 — VCP Detector (2-window with 30% contraction rule)

In [7]:
def detect_vcp(
    df: pd.DataFrame,
    swings: list[SwingPoint],
    min_base_duration: int = 20,
    max_retrace: float = 0.50,
    min_window_bars: int = 5,
    min_window_gap: int = 3,
    max_contraction: float = 0.70,
    contraction_tail: int = 5,
) -> list[dict]:
    """
    VCP detector (2-window):
      - W1: peak to first pullback (min_window_bars)
      - W2: trailing window ending at candidate end
      - Stage-2 proxy: sma50 > sma200
      - No close below sma200 during base
      - No breakout above peak + 1*ATR
      - Retrace <= max_retrace
      - Duration >= min_base_duration
      - Contraction: W2_range <= max_contraction * W1_range
    Returns one candidate per peak (latest valid end).
    """
    highs = df["high"].values
    lows = df["low"].values
    closes = df["close"].values
    sma50 = df["sma50"].values
    sma200 = df["sma200"].values
    atr = df["atr14"].values
    n = len(df)

    peaks = sorted([s for s in swings if s.type == "HIGH"], key=lambda s: s.index)
    out: list[dict] = []

    min_t = max(min_base_duration - 1, 2 * min_window_bars + min_window_gap - 1)

    for pk in peaks:
        p = pk.index
        peak_price = float(pk.price)
        if p >= n - min_t:
            continue

        w1_start = p
        w1_end = p + min_window_bars - 1
        if w1_end >= n:
            continue

        w1_low = float(np.min(lows[w1_start : w1_end + 1]))
        w1_high = float(np.max(highs[w1_start : w1_end + 1]))
        w1_range = max(w1_high - w1_low, 1e-9)

        best = None
        t_start = max(p + min_t, w1_end + min_window_gap + min_window_bars)

        for t in range(t_start, n):
            w2_end = t
            w2_start = t - min_window_bars + 1
            if w2_start <= w1_end + min_window_gap:
                continue

            if not all(np.isfinite(v) for v in [sma50[t], sma200[t]]):
                continue
            if sma50[t] <= sma200[t]:
                continue

            span_high = float(np.max(highs[p : t + 1]))
            span_low = float(np.min(lows[p : t + 1]))
            retrace = (peak_price - span_low) / peak_price if peak_price > 0 else 1.0
            if retrace > max_retrace:
                continue

            atr_ref = atr[p] if np.isfinite(atr[p]) and atr[p] > 0 else (atr[t] if np.isfinite(atr[t]) and atr[t] > 0 else 0.0)
            if span_high > peak_price + 1.0 * atr_ref:
                continue

            valid_sma200 = np.all(np.isfinite(sma200[p : t + 1]))
            if not valid_sma200:
                continue
            if not np.all(closes[p : t + 1] >= sma200[p : t + 1]):
                continue

            combined_duration = t - w1_start + 1
            if combined_duration < min_base_duration:
                continue

            w2_low = float(np.min(lows[w2_start : w2_end + 1]))
            w2_high = float(np.max(highs[w2_start : w2_end + 1]))
            w2_range = max(w2_high - w2_low, 1e-9)

            ratio = w2_range / w1_range
            if ratio > max_contraction:
                continue

            mid = max(p + 1, t - contraction_tail + 1)
            c_high = float(np.max(highs[mid : t + 1]))
            c_low = float(np.min(lows[mid : t + 1]))

            best = {
                "kind": "VCP",
                "peak_idx": int(p),
                "peak_price": round(peak_price, 2),
                "start_idx": int(w1_start),
                "mid_idx": int(mid),
                "end_idx": int(t),
                "w1_start_idx": int(w1_start),
                "w1_end_idx": int(w1_end),
                "w2_start_idx": int(w2_start),
                "w2_end_idx": int(w2_end),
                "w1_range": round(w1_range, 3),
                "w2_range": round(w2_range, 3),
                "w2_w1_ratio": round(ratio, 4),
                "base_high": round(span_high, 2),
                "base_low": round(span_low, 2),
                "contraction_high": round(c_high, 2),
                "contraction_low": round(c_low, 2),
                "retrace_pct": round(retrace, 4),
                "duration": int(combined_duration),
            }

        if best is not None:
            out.append(best)

    return out


vcp_raw = detect_vcp(df, major_swings, min_base_duration=20, max_retrace=0.50, max_contraction=0.70)
print(f"VCP candidates: {len(vcp_raw)}")
for v in vcp_raw[:5]:
    print(f"  peak={v['peak_price']} W1={v['w1_range']:.2f} W2={v['w2_range']:.2f} ratio={v['w2_w1_ratio']:.3f} dur={v['duration']}")

VCP candidates: 5
  peak=11.25 W1=2.68 W2=1.23 ratio=0.459 dur=46
  peak=10.2 W1=2.74 W2=1.23 ratio=0.449 dur=27
  peak=15.42 W1=3.67 W2=2.32 ratio=0.632 dur=49
  peak=40.2 W1=11.11 W2=6.05 ratio=0.545 dur=26
  peak=34.86 W1=11.97 W2=7.21 ratio=0.603 dur=20


## 7 — PIVOT Detector (base/VCP-gated)

In [8]:
def detect_pivots(
    df: pd.DataFrame,
    base_regions: list[dict],
    vcp_regions: list[dict],
    vol_avg_window: int = 20,
    pivot_window: int = 3,
    max_range_pct: float = 0.08,
) -> list[dict]:
    """
    PIVOT:
      - Must sit inside valid BASE or VCP region
      - Last 3 bars each volume < 20-day average volume
      - 3-bar range: (maxHigh3 - minLow3) / minLow3 <= 8%
    Returns one latest pivot per parent region.
    """
    if len(df) < max(vol_avg_window, pivot_window):
        return []

    highs = df["high"].values
    lows = df["low"].values
    closes = df["close"].values
    vols = df["volume"].values
    vol_avg = df["vol_avg20"].values if "vol_avg20" in df.columns else pd.Series(vols).rolling(vol_avg_window).mean().values

    regions = (
        [{**r, "parent": "BASE"} for r in base_regions]
        + [{**r, "parent": "VCP"} for r in vcp_regions]
    )

    best_per_region: dict[tuple[str, int], dict] = {}

    for r in regions:
        s_idx = int(r["start_idx"])
        e_idx = int(r["end_idx"])
        if e_idx - s_idx + 1 < pivot_window:
            continue

        for t in range(max(s_idx + pivot_window - 1, vol_avg_window), e_idx + 1):
            ws = t - pivot_window + 1

            rng_high = float(np.max(highs[ws : t + 1]))
            rng_low = float(np.min(lows[ws : t + 1]))
            range_pct = (rng_high - rng_low) / max(rng_low, 1e-9)
            if range_pct > max_range_pct:
                continue

            avg_vals = vol_avg[ws : t + 1]
            if len(avg_vals) < pivot_window or not np.all(np.isfinite(avg_vals)):
                continue
            vols3 = vols[ws : t + 1]
            if not bool(np.all(vols3 < avg_vals)):
                continue

            piv = {
                "kind": "PIVOT",
                "parent": r["parent"],
                "pivot_idx": int(t),
                "start_idx": int(ws),
                "end_idx": int(t),
                "pivot_price": round(float(rng_high), 2),
                "range_pct": round(range_pct, 4),
                "vol_3": [int(v) for v in vols3],
                "vol_avg_3": [round(float(v), 1) for v in avg_vals],
                "region_peak_idx": int(r["peak_idx"]),
            }

            key = (r["parent"], r["peak_idx"])
            cur = best_per_region.get(key)
            if cur is None or piv["pivot_idx"] > cur["pivot_idx"]:
                best_per_region[key] = piv

    return sorted(best_per_region.values(), key=lambda x: x["pivot_idx"])


pivot_signals = detect_pivots(df, base_raw, vcp_raw, vol_avg_window=20, pivot_window=3, max_range_pct=0.08)
print(f"PIVOT signals: {len(pivot_signals)}")
for p in pivot_signals[:5]:
    print(f"  parent={p['parent']} idx={p['pivot_idx']} price={p['pivot_price']} range={p['range_pct']:.4f}")

PIVOT signals: 2
  parent=BASE idx=348 price=9.69 range=0.0743
  parent=VCP idx=348 price=9.69 range=0.0743


## 8 — Helpers: Swing Split + Region Merge

In [9]:
def split_swings_for_chart(
    swings: list[SwingPoint],
    df_ctx: pd.DataFrame,
    atr_buffer: float = 1.0,
) -> tuple[list[SwingPoint], list[SwingPoint], list[SwingPoint]]:
    """Return (active, expired_high, expired_low)."""
    active, exp_h, exp_l = [], [], []
    highs = df_ctx["high"].values
    lows = df_ctx["low"].values
    atr_arr = df_ctx["atr14"].values

    for s in swings:
        if s.index >= len(df_ctx) - 1:
            active.append(s)
            continue

        atr_ref = s.atr if np.isfinite(s.atr) and s.atr > 0 else atr_arr[s.index]
        if not np.isfinite(atr_ref) or atr_ref <= 0:
            atr_ref = atr_arr[-1] if np.isfinite(atr_arr[-1]) and atr_arr[-1] > 0 else 0.0

        thr = atr_buffer * atr_ref
        if s.type == "HIGH":
            if np.any(highs[s.index + 1 :] > s.price + thr):
                exp_h.append(s)
            else:
                active.append(s)
        else:
            if np.any(lows[s.index + 1 :] < s.price - thr):
                exp_l.append(s)
            else:
                active.append(s)

    return active, exp_h, exp_l


def merge_regions(candidates: list[dict], df: pd.DataFrame, contraction_tail: int = 5) -> list[dict]:
    """Merge overlapping base/vcp regions. One rep per peak (latest end). 60% time+price overlap = merge."""
    if not candidates:
        return []

    by_peak: dict[int, dict] = {}
    for c in candidates:
        pk = c["peak_idx"]
        cur = by_peak.get(pk)
        if cur is None or c["end_idx"] > cur["end_idx"]:
            by_peak[pk] = c

    reps = sorted(by_peak.values(), key=lambda x: x["start_idx"])
    merged: list[dict] = []
    highs = df["high"].values
    lows = df["low"].values

    for c in reps:
        if not merged:
            merged.append(c.copy())
            continue

        last = merged[-1]
        t0 = max(last["start_idx"], c["start_idx"])
        t1 = min(last["end_idx"], c["end_idx"])
        time_ov = max(0, t1 - t0 + 1)
        min_len = max(1, min(last["end_idx"] - last["start_idx"] + 1, c["end_idx"] - c["start_idx"] + 1))
        t_ratio = time_ov / min_len

        p0 = max(last["base_low"], c["base_low"])
        p1 = min(last["base_high"], c["base_high"])
        price_ov = max(0.0, p1 - p0)
        min_h = max(1e-9, min(last["base_high"] - last["base_low"], c["base_high"] - c["base_low"]))
        p_ratio = price_ov / min_h

        if t_ratio >= 0.60 and p_ratio >= 0.60:
            ns = min(last["start_idx"], c["start_idx"])
            ne = max(last["end_idx"], c["end_idx"])
            nm = max(ns + 1, ne - contraction_tail + 1)
            b_high = float(np.max(highs[ns : ne + 1]))
            b_low = float(np.min(lows[ns : ne + 1]))
            merged[-1] = {
                **last,
                "start_idx": int(ns),
                "end_idx": int(ne),
                "base_high": round(b_high, 2),
                "base_low": round(b_low, 2),
                "duration": int(ne - ns + 1),
                "retrace_pct": min(last.get("retrace_pct", 1.0), c.get("retrace_pct", 1.0)),
            }
        else:
            merged.append(c.copy())

    return merged


base_merged = merge_regions(base_raw, df)
vcp_merged = merge_regions(vcp_raw, df)
print(f"BASE merged: {len(base_merged)} | VCP merged: {len(vcp_merged)}")

BASE merged: 5 | VCP merged: 4


## 9 — Final Chart

In [10]:
maj_act, maj_xh, maj_xl = split_swings_for_chart(major_swings, df)
min_act, min_xh, min_xl = split_swings_for_chart(minor_swings, df)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.76, 0.24], vertical_spacing=0.03)

fig.add_trace(go.Candlestick(
    x=df["date"], open=df["open"], high=df["high"],
    low=df["low"], close=df["close"], name="Price",
), row=1, col=1)

fig.add_trace(go.Scatter(x=df["date"], y=df["ema20"], name="EMA 20", line=dict(color="#f59e0b", width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=df["date"], y=df["sma50"], name="SMA 50", line=dict(color="#3b82f6", width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=df["date"], y=df["sma200"], name="SMA 200", line=dict(color="#ef4444", width=1.5)), row=1, col=1)

# Major swings
for arr, name, symbol, color, sz in [
    ([s for s in maj_act if s.type == "HIGH"], "Major High", "star", "#dc2626", 10),
    ([s for s in maj_act if s.type == "LOW"], "Major Low", "star", "#16a34a", 10),
]:
    if arr:
        fig.add_trace(go.Scatter(
            x=[df["date"].iloc[s.index] for s in arr],
            y=[s.price for s in arr],
            mode="markers", name=name,
            marker=dict(symbol=symbol, size=sz, color=color, line=dict(width=1, color="white")),
        ), row=1, col=1)

# Minor swings (legend-only by default)
for arr, name, symbol, color, sz in [
    ([s for s in min_act if s.type == "HIGH"], "Minor High", "triangle-down", "rgba(244,63,94,0.6)", 6),
    ([s for s in min_act if s.type == "LOW"], "Minor Low", "triangle-up", "rgba(34,197,94,0.6)", 6),
]:
    if arr:
        fig.add_trace(go.Scatter(
            x=[df["date"].iloc[s.index] for s in arr],
            y=[s.price for s in arr],
            mode="markers", name=name,
            marker=dict(symbol=symbol, size=sz, color=color),
            visible="legendonly",
        ), row=1, col=1)

# BASE rectangles (green)
for b in base_merged[-20:]:
    fig.add_shape(
        type="rect",
        x0=df["date"].iloc[b["start_idx"]], x1=df["date"].iloc[b["end_idx"]],
        y0=b["base_low"], y1=b["base_high"],
        xref="x", yref="y",
        line=dict(color="rgba(34,197,94,0.55)", width=1),
        fillcolor="rgba(34,197,94,0.08)",
        layer="below",
    )

# VCP rectangles (blue) + contraction tail
for v in vcp_merged[-20:]:
    fig.add_shape(
        type="rect",
        x0=df["date"].iloc[v["start_idx"]], x1=df["date"].iloc[v["end_idx"]],
        y0=v["base_low"], y1=v["base_high"],
        xref="x", yref="y",
        line=dict(color="rgba(59,130,246,0.55)", width=1),
        fillcolor="rgba(59,130,246,0.06)",
        layer="below",
    )
    if "contraction_low" in v:
        fig.add_shape(
            type="rect",
            x0=df["date"].iloc[v["mid_idx"]], x1=df["date"].iloc[v["end_idx"]],
            y0=v["contraction_low"], y1=v["contraction_high"],
            xref="x", yref="y",
            line=dict(color="rgba(37,99,235,0.75)", width=1),
            fillcolor="rgba(37,99,235,0.15)",
            layer="below",
        )

# PIVOT markers
if pivot_signals:
    fig.add_trace(go.Scatter(
        x=[df["date"].iloc[p["pivot_idx"]] for p in pivot_signals],
        y=[p["pivot_price"] for p in pivot_signals],
        mode="markers", name="Pivot (3d low-vol tight)",
        marker=dict(symbol="diamond", size=11, color="#fbbf24", line=dict(width=1.5, color="white")),
    ), row=1, col=1)

# Volume
vol_colors = ["#22c55e" if df["close"].iloc[i] >= df["open"].iloc[i] else "#ef4444" for i in range(len(df))]
fig.add_trace(go.Bar(x=df["date"], y=df["volume"], name="Volume", marker_color=vol_colors, opacity=0.45), row=2, col=1)

fig.update_layout(
    title=f"{TICKER} — BASE / VCP / PIVOT",
    template="plotly_dark",
    height=900,
    xaxis_rangeslider_visible=False,
    legend=dict(orientation="h", y=1.02, x=0),
)
fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text="Volume", row=2, col=1)
fig.show(renderer="notebook_connected")

## 10 — Validation Report

In [11]:
print("=" * 60)
print("VALIDATION REPORT")
print("=" * 60)

# ---- Swing tiers ----
print(f"\nSwing tiers:")
print(f"  Major: {len(major_swings)} ({sum(1 for s in major_swings if s.type=='HIGH')} H, {sum(1 for s in major_swings if s.type=='LOW')} L)")
print(f"  Minor: {len(minor_swings)} ({sum(1 for s in minor_swings if s.type=='HIGH')} H, {sum(1 for s in minor_swings if s.type=='LOW')} L)")

# ---- BASE validation ----
print(f"\nBASE candidates: {len(base_raw)} (merged: {len(base_merged)})")
base_ok = True
for i, b in enumerate(base_raw):
    r_ok = b["retrace_pct"] < 0.50
    d_ok = b["duration"] > 20
    brk_ok = b["base_high"] <= b["peak_price"]
    row_ok = r_ok and d_ok and brk_ok
    if not row_ok:
        base_ok = False
    print(f"  [{i}] peak={b['peak_price']} retrace={b['retrace_pct']:.3f}({'OK' if r_ok else 'FAIL'}) dur={b['duration']}({'OK' if d_ok else 'FAIL'}) no_breakout={'OK' if brk_ok else 'FAIL'}")
print(f"  BASE rules: {'ALL PASS' if base_ok else 'SOME FAIL'}")

# ---- VCP validation ----
print(f"\nVCP candidates: {len(vcp_raw)} (merged: {len(vcp_merged)})")
vcp_ok = True
for i, v in enumerate(vcp_raw):
    c_ok = v["w2_w1_ratio"] <= 0.70
    r_ok = v["retrace_pct"] <= 0.50
    row_ok = c_ok and r_ok
    if not row_ok:
        vcp_ok = False
    print(f"  [{i}] peak={v['peak_price']} W1={v['w1_range']:.2f} W2={v['w2_range']:.2f} ratio={v['w2_w1_ratio']:.3f}({'OK' if c_ok else 'FAIL'}) retrace={v['retrace_pct']:.3f}({'OK' if r_ok else 'FAIL'})")
print(f"  VCP contraction rule: {'ALL PASS' if vcp_ok else 'SOME FAIL'}")

# ---- PIVOT validation ----
print(f"\nPIVOT signals: {len(pivot_signals)}")
piv_ok = True
for i, p in enumerate(pivot_signals):
    rng_ok = p["range_pct"] <= 0.08
    vol_ok = all(v < a for v, a in zip(p["vol_3"], p["vol_avg_3"]))
    row_ok = rng_ok and vol_ok
    if not row_ok:
        piv_ok = False
    print(f"  [{i}] parent={p['parent']} idx={p['pivot_idx']} range={p['range_pct']:.4f}({'OK' if rng_ok else 'FAIL'}) low_vol={'OK' if vol_ok else 'FAIL'}")
print(f"  PIVOT rules: {'ALL PASS' if piv_ok else 'SOME FAIL'}")

print("\n" + "=" * 60)
overall = base_ok and vcp_ok and piv_ok
print(f"OVERALL: {'ALL RULES PASS' if overall else 'SOME RULES FAIL — review above'}")
print("=" * 60)

VALIDATION REPORT

Swing tiers:
  Major: 45 (22 H, 23 L)
  Minor: 109 (58 H, 51 L)

BASE candidates: 6 (merged: 5)
  [0] peak=7.23 retrace=0.456(OK) dur=27(OK) no_breakout=OK
  [1] peak=6.88 retrace=0.477(OK) dur=33(OK) no_breakout=OK
  [2] peak=11.25 retrace=0.456(OK) dur=50(OK) no_breakout=OK
  [3] peak=10.2 retrace=0.400(OK) dur=29(OK) no_breakout=OK
  [4] peak=15.42 retrace=0.415(OK) dur=46(OK) no_breakout=OK
  [5] peak=40.2 retrace=0.489(OK) dur=26(OK) no_breakout=OK
  BASE rules: ALL PASS

VCP candidates: 5 (merged: 4)
  [0] peak=11.25 W1=2.68 W2=1.23 ratio=0.459(OK) retrace=0.456(OK)
  [1] peak=10.2 W1=2.74 W2=1.23 ratio=0.449(OK) retrace=0.400(OK)
  [2] peak=15.42 W1=3.67 W2=2.32 ratio=0.632(OK) retrace=0.415(OK)
  [3] peak=40.2 W1=11.11 W2=6.05 ratio=0.545(OK) retrace=0.489(OK)
  [4] peak=34.86 W1=11.97 W2=7.21 ratio=0.603(OK) retrace=0.371(OK)
  VCP contraction rule: ALL PASS

PIVOT signals: 2
  [0] parent=BASE idx=348 range=0.0743(OK) low_vol=OK
  [1] parent=VCP idx=348 rang